In [1]:
import subprocess, sys
for pkg_name, import_name in [('xgboost','xgboost'), ('lightgbm','lightgbm')]:
    try:
        __import__(import_name); print(f'{import_name} ya disponible ✅')
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install',pkg_name,'--quiet'])
        print(f'{pkg_name} instalado ✅ — REINICIAR KERNEL antes de continuar')
print('Listo — continuar con celda 0')



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


xgboost instalado ✅ — REINICIAR KERNEL antes de continuar
lightgbm instalado ✅ — REINICIAR KERNEL antes de continuar
Listo — continuar con celda 0



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [1]:
import os, sys, time, pickle, warnings, gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)
from itertools import product
import mlflow
warnings.filterwarnings('ignore')

SAVE_DIR = '/srv/work'
IN_FEAT  = f'{SAVE_DIR}/te_features_v5.parquet'
OUT_DIR  = f'{SAVE_DIR}/models'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
HORIZON      = 2
LABEL_COLS   = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}
ROUTE_KEYS   = ['source_agent', 'cloud_provider', 'region']
ID_COLS      = ROUTE_KEYS + ['window']
TARGET_COL   = 'label_ventana'
SPLIT_COL    = 'split'

UMBRALES = {
    'qwk':        {'floor': 0.65, 'target': 0.72, 'stretch': 0.80},
    'recall_alta':{'floor': 0.55, 'target': 0.75, 'stretch': 0.85},
    'macro_f1':   {'floor': 0.55, 'target': 0.65, 'stretch': 0.75},
    'bal_acc':    {'floor': 0.60, 'target': 0.70, 'stretch': 0.78},
}

try:
    mlflow.set_tracking_uri('http://localhost:6000')
    mlflow.set_experiment('latam2cloud')
    MLFLOW_OK = True; print('MLflow OK — http://localhost:6000')
except Exception:
    MLFLOW_OK = False; print('MLflow no disponible — entrenamiento continúa sin tracking')

print(f'XGBoost {xgb.__version__}  |  LightGBM {lgb.__version__}')
print(f'Input : {IN_FEAT}')
print(f'Output: {OUT_DIR}')


MLflow OK — http://localhost:6000
XGBoost 3.2.0  |  LightGBM 4.7.0
Input : /srv/work/te_features_v5.parquet
Output: /srv/work/models


In [2]:
import gc

print('Cargando te_features_v5.parquet...')
df = pq.read_table(IN_FEAT).to_pandas()
print(f'  Shape: {df.shape}')

df['y'] = df.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df = df.dropna(subset=['y'])
df['y'] = df['y'].astype(int)
print(f'  Post target-shift: {df.shape}')

feat_cols_v5 = [c for c in df.columns
                if c not in ID_COLS + [TARGET_COL, SPLIT_COL, 'y']
                and not c.startswith('n_samp')]
print(f'  Features: {len(feat_cols_v5)}')

train_mask = df['split'] == 'train'
val_mask   = df['split'] == 'val'
y_train    = df.loc[train_mask, 'y'].values.astype(int)
y_val      = df.loc[val_mask,   'y'].values.astype(int)

n_total         = len(y_train)
class_counts    = np.bincount(y_train, minlength=4)
sqrt_w_arr      = np.sqrt(n_total / (4 * class_counts)).astype('float32')
sample_weight_train = sqrt_w_arr[y_train]

print('\nDistribucion train + pesos sqrt_w_arr:')
for k, name in LABEL_COLS.items():
    print(f'  {name:<8}: n={class_counts[k]:>9,}  peso={sqrt_w_arr[k]:.3f}')
print(f'  Ratio Alta/Normal: {sqrt_w_arr[3]/sqrt_w_arr[0]:.1f}×')

fill0_cols  = [c for c in feat_cols_v5 if c.startswith('std_') or c.startswith('bgp_')]
other_fill0 = [c for c in feat_cols_v5
               if not c.startswith('lag') and 'ratio' not in c and c not in fill0_cols]
fill_cols_set = set(fill0_cols + other_fill0)
fill_idx = [i for i, c in enumerate(feat_cols_v5) if c in fill_cols_set]

print('\nExtrayendo X_train_xgb (float32)...')
X_train_xgb = df.loc[train_mask, feat_cols_v5].values.astype('float32')
for idx in fill_idx:
    m = np.isnan(X_train_xgb[:, idx])
    if m.any(): X_train_xgb[m, idx] = 0.0
gc.collect()

print('Extrayendo X_val_xgb (float32)...')
X_val_xgb = df.loc[val_mask, feat_cols_v5].values.astype('float32')
for idx in fill_idx:
    m = np.isnan(X_val_xgb[:, idx])
    if m.any(): X_val_xgb[m, idx] = 0.0
gc.collect()

print(f'\n  X_train_xgb: {X_train_xgb.shape}  X_val_xgb: {X_val_xgb.shape}')
print(f'  train={len(y_train):,}  val={len(y_val):,}')
print('Celda 1 OK')


Cargando te_features_v5.parquet...
  Shape: (3763917, 109)
  Post target-shift: (3761847, 110)
  Features: 100

Distribucion train + pesos sqrt_w_arr:
  Normal  : n=1,899,347  peso=0.529
  Baja    : n=  153,319  peso=1.862
  Media   : n=   36,421  peso=3.820
  Alta    : n=   36,691  peso=3.806
  Ratio Alta/Normal: 7.2×

Extrayendo X_train_xgb (float32)...
Extrayendo X_val_xgb (float32)...

  X_train_xgb: (2125778, 100)  X_val_xgb: (644690, 100)
  train=2,125,778  val=644,690
Celda 1 OK


In [3]:
def evaluar(nombre, y_true, y_prob):
    y_pred    = np.argmax(y_prob, axis=1)
    n_alta    = (y_true == 3).sum()
    tp_alta   = ((y_pred == 3) & (y_true == 3)).sum()
    fp_alta   = ((y_pred == 3) & (y_true != 3)).sum()
    rec_alta  = tp_alta / n_alta if n_alta > 0 else 0.0
    prec_alta = tp_alta / (tp_alta + fp_alta) if (tp_alta + fp_alta) > 0 else 0.0
    qwk       = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    macro_f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bal_acc   = balanced_accuracy_score(y_true, y_pred)
    mae_ord   = np.mean(np.abs(y_pred - y_true))
    cm        = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    cand      = (qwk >= UMBRALES['qwk']['floor'] and
                 rec_alta >= UMBRALES['recall_alta']['floor'])

    sep = '═' * 110
    print(f'\n{sep}')
    print(f'  {nombre} — Validacion')
    print(f'  QWK={qwk:.4f}  MacroF1={macro_f1:.4f}  BalAcc={bal_acc:.4f}'
          f'  MAEord={mae_ord:.4f}  RecAlta={rec_alta:.4f}  PrecAlta={prec_alta:.4f}')
    flag = '✅ CANDIDATO H2' if cand else '❌ No candidato'
    print(f'  {flag}  (QWK≥0.65={qwk>=0.65}  RecAlta≥0.55={rec_alta>=0.55})')
    print(f'{sep}')
    hdr = f'          {"Normal":>8} {"Baja":>8} {"Media":>8} {"Alta":>8}'
    print(f'  Confusion matrix (filas=real, cols=pred):\n{hdr}')
    for i, row in enumerate(cm):
        print(f'  {LABEL_COLS[i]:<8}' + ''.join(f'{v:>9,}' for v in row))
    return dict(qwk=qwk, recall_alta=rec_alta, prec_alta=prec_alta,
                macro_f1=macro_f1, bal_acc=bal_acc, mae_ord=mae_ord,
                candidato=cand, confusion_matrix=cm, y_prob=y_prob)

refs = {
    'rf_v5':       dict(qwk=0.6420, recall_alta=0.6726, macro_f1=0.6560, bal_acc=0.6720),
    'xgb_v5':      dict(qwk=0.6232, recall_alta=0.6648, macro_f1=0.6511, bal_acc=0.6985),
    'lgb_v5':      dict(qwk=0.6300, recall_alta=0.6824, macro_f1=0.6533, bal_acc=0.7025),
    'ens_rfxgb_v6':dict(qwk=0.6413, recall_alta=0.5605, macro_f1=0.6563, bal_acc=0.6164),
}
print('Función evaluar lista.')
print(f'  Mejor resultado previo: RF v5 QWK=0.6420 | Ensemble RF+XGB_v6 QWK=0.6413')


Función evaluar lista.
  Mejor resultado previo: RF v5 QWK=0.6420 | Ensemble RF+XGB_v6 QWK=0.6413


In [4]:
print('Cargando RF v5 desde disco...')
t0 = time.time()
with open(f'{OUT_DIR}/rf_v5_model.pkl', 'rb') as f:
    rf_v5 = pickle.load(f)
print(f'  RF v5 cargado en {time.time()-t0:.1f}s')

ratio_cols_rf = [c for c in feat_cols_v5 if 'ratio' in c]
other_nan_rf  = [c for c in feat_cols_v5 if c not in ratio_cols_rf]
X_val_imp = df.loc[val_mask, feat_cols_v5].copy().astype('float32')
X_val_imp[ratio_cols_rf] = X_val_imp[ratio_cols_rf].fillna(1.0)
X_val_imp[other_nan_rf]  = X_val_imp[other_nan_rf].fillna(0.0)

with open(f'{OUT_DIR}/scaler_v5.pkl', 'rb') as f:
    scaler = pickle.load(f)
X_val_sc = scaler.transform(X_val_imp.values)
del X_val_imp; gc.collect()

print('  Prediciendo en val (chunks 50k)...')
t0 = time.time()
CHUNK = 50_000
n_val = X_val_sc.shape[0]
y_prob_rf = np.zeros((n_val, 4), dtype='float32')
for i in range(0, n_val, CHUNK):
    y_prob_rf[i:i+CHUNK] = rf_v5.predict_proba(X_val_sc[i:i+CHUNK])
np.save(f'{OUT_DIR}/rf_v5_val_proba.npy', y_prob_rf)
print(f'  Listo en {time.time()-t0:.1f}s')

del rf_v5, X_val_sc; gc.collect()
res_rf = evaluar('RF v5 (cargado, D31)', y_val, y_prob_rf)


Cargando RF v5 desde disco...
  RF v5 cargado en 42.9s
  Prediciendo en val (chunks 50k)...


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    0.8s
[Parallel(n_jobs=4)]: Done 200 out of 200 | elapsed:    0.8s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    0.8s
[Parallel(n_jobs=4)]: Done 200 out of 200 | elapsed:    0.8s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    0.8s
[Parallel(n_jobs=4)]: Done 200 out of 200 | elapsed:    0.8s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done  42 tasks      |

  Listo en 11.4s

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  RF v5 (cargado, D31) — Validacion
  QWK=0.6420  MacroF1=0.6560  BalAcc=0.6720  MAEord=0.1594  RecAlta=0.6726  PrecAlta=0.5381
  ❌ No candidato  (QWK≥0.65=False  RecAlta≥0.55=True)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Confusion matrix (filas=real, cols=pred):
            Normal     Baja    Media     Alta
  Normal    520,331   18,017    4,760    8,083
  Baja       16,261   39,146    1,824    1,414
  Media       4,782    1,925    5,962    2,100
  Alta        4,642      593    1,340   13,510


In [5]:

print('Entrenando XGBoost v7 (D32 — sample_weight=sqrt_w_arr)...')
print(f'  n_estimators=2000  |  lr=0.05  |  colsample=0.6  |  early_stop=100')
print(f'  Peso Alta/Normal: {sqrt_w_arr[3]/sqrt_w_arr[0]:.1f}×')

t0 = time.time()
xgb_v7 = xgb.XGBClassifier(
    n_estimators          = 2000,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.6,
    min_child_weight      = 5,
    gamma                 = 0.1,
    objective             = 'multi:softprob',
    num_class             = 4,
    early_stopping_rounds = 100,
    eval_metric           = 'mlogloss',
    n_jobs                = 4,
    tree_method           = 'hist',
    random_state          = RANDOM_STATE,
    verbosity             = 1,
)
xgb_v7.fit(
    X_train_xgb, y_train,
    sample_weight         = sample_weight_train,
    eval_set              = [(X_val_xgb, y_val)],
    verbose               = 100,
)
elapsed   = time.time() - t0
best_iter = xgb_v7.best_iteration
print(f'\nTiempo: {elapsed/60:.1f} min  |  best_iteration: {best_iter}')

y_prob_xgb = xgb_v7.predict_proba(X_val_xgb)
np.save(f'{OUT_DIR}/xgb_v7_val_proba.npy', y_prob_xgb)
xgb_v7.save_model(f'{OUT_DIR}/xgb_v7_model.json')

res_xgb = evaluar('XGBoost v7 (D32 — sample_weight)', y_val, y_prob_xgb)

for k in ('qwk','recall_alta'):
    v5 = refs['xgb_v5'][k]; v7 = res_xgb[k]
    print(f'  Δ {k}: v5={v5:.4f}  v7={v7:.4f}  ({v7-v5:+.4f})')

if MLFLOW_OK:
    with mlflow.start_run(run_name='xgb_v7'):
        mlflow.log_params(dict(n_estimators=2000, early_stopping=100,
                               best_iteration=best_iter, sample_weight='sqrt_w_arr',
                               decision='D32'))
        mlflow.log_metrics(dict(qwk=res_xgb['qwk'], recall_alta=res_xgb['recall_alta'],
                                macro_f1=res_xgb['macro_f1'], bal_acc=res_xgb['bal_acc']))


Entrenando XGBoost v7 (D32 — sample_weight=sqrt_w_arr)...
  n_estimators=2000  |  lr=0.05  |  colsample=0.6  |  early_stop=100
  Peso Alta/Normal: 7.2×
[0]	validation_0-mlogloss:0.64735
[100]	validation_0-mlogloss:0.33764
[200]	validation_0-mlogloss:0.33307
[300]	validation_0-mlogloss:0.32999
[400]	validation_0-mlogloss:0.32772
[500]	validation_0-mlogloss:0.32609
[600]	validation_0-mlogloss:0.32448
[700]	validation_0-mlogloss:0.32319
[800]	validation_0-mlogloss:0.32209
[900]	validation_0-mlogloss:0.32118
[1000]	validation_0-mlogloss:0.32025
[1100]	validation_0-mlogloss:0.31937
[1200]	validation_0-mlogloss:0.31864
[1300]	validation_0-mlogloss:0.31807
[1400]	validation_0-mlogloss:0.31739
[1500]	validation_0-mlogloss:0.31665
[1600]	validation_0-mlogloss:0.31604
[1700]	validation_0-mlogloss:0.31543
[1800]	validation_0-mlogloss:0.31495
[1900]	validation_0-mlogloss:0.31435
[1999]	validation_0-mlogloss:0.31380

Tiempo: 32.3 min  |  best_iteration: 1999

═══════════════════════════════════════

In [6]:

print('Entrenando LightGBM v7 (D33 — sample_weight=sqrt_w_arr)...')
print(f'  num_boost_round=2000  |  lr=0.05  |  num_leaves=63  |  early_stop=100')
print(f'  Peso Alta/Normal: {sqrt_w_arr[3]/sqrt_w_arr[0]:.1f}×')

dtrain_lgb = lgb.Dataset(X_train_xgb, label=y_train,
                          weight=sample_weight_train,
                          feature_name=feat_cols_v5,
                          free_raw_data=True)
dval_lgb = lgb.Dataset(X_val_xgb, label=y_val,
                        reference=dtrain_lgb,
                        free_raw_data=True)

lgb_params = dict(
    objective        = 'multiclass',
    num_class        = 4,
    metric           = 'multi_logloss',
    num_leaves       = 63,
    learning_rate    = 0.05,
    feature_fraction = 0.6,
    bagging_fraction = 0.8,
    bagging_freq     = 5,
    min_child_samples= 20,
    n_jobs           = 4,
    random_state     = RANDOM_STATE,
    verbose          = -1,
)

t0 = time.time()
lgb_v7 = lgb.train(
    lgb_params, dtrain_lgb,
    num_boost_round = 2000,
    valid_sets      = [dval_lgb],
    callbacks       = [lgb.early_stopping(100), lgb.log_evaluation(100)],
)
elapsed   = time.time() - t0
best_iter = lgb_v7.best_iteration
print(f'\nTiempo: {elapsed/60:.1f} min  |  best_iteration: {best_iter}')

y_prob_lgb = lgb_v7.predict(X_val_xgb)
np.save(f'{OUT_DIR}/lgb_v7_val_proba.npy', y_prob_lgb)
lgb_v7.save_model(f'{OUT_DIR}/lgb_v7_model.txt')

res_lgb = evaluar('LightGBM v7 (D33 — sample_weight)', y_val, y_prob_lgb)

for k in ('qwk','recall_alta'):
    v5 = refs['lgb_v5'][k]; v7 = res_lgb[k]
    print(f'  Δ {k}: v5={v5:.4f}  v7={v7:.4f}  ({v7-v5:+.4f})')

if MLFLOW_OK:
    with mlflow.start_run(run_name='lgb_v7'):
        mlflow.log_params(dict(num_boost_round=2000, early_stopping=100,
                               best_iteration=best_iter, sample_weight='sqrt_w_arr',
                               decision='D33'))
        mlflow.log_metrics(dict(qwk=res_lgb['qwk'], recall_alta=res_lgb['recall_alta'],
                                macro_f1=res_lgb['macro_f1'], bal_acc=res_lgb['bal_acc']))


Entrenando LightGBM v7 (D33 — sample_weight=sqrt_w_arr)...
  num_boost_round=2000  |  lr=0.05  |  num_leaves=63  |  early_stop=100
  Peso Alta/Normal: 7.2×
Training until validation scores don't improve for 100 rounds
[100]	valid_0's multi_logloss: 0.333302
[200]	valid_0's multi_logloss: 0.328595
[300]	valid_0's multi_logloss: 0.325718
[400]	valid_0's multi_logloss: 0.323333
[500]	valid_0's multi_logloss: 0.321174
[600]	valid_0's multi_logloss: 0.319429
[700]	valid_0's multi_logloss: 0.317572
[800]	valid_0's multi_logloss: 0.316095
[900]	valid_0's multi_logloss: 0.314612
[1000]	valid_0's multi_logloss: 0.312898
[1100]	valid_0's multi_logloss: 0.311525
[1200]	valid_0's multi_logloss: 0.310278
[1300]	valid_0's multi_logloss: 0.308847
[1400]	valid_0's multi_logloss: 0.307611
[1500]	valid_0's multi_logloss: 0.306344
[1600]	valid_0's multi_logloss: 0.305096
[1700]	valid_0's multi_logloss: 0.304028
[1800]	valid_0's multi_logloss: 0.30308
[1900]	valid_0's multi_logloss: 0.301935
[2000]	valid_

In [7]:

p_rf  = np.load(f'{OUT_DIR}/rf_v5_val_proba.npy')
p_xgb = np.load(f'{OUT_DIR}/xgb_v7_val_proba.npy')
p_lgb = np.load(f'{OUT_DIR}/lgb_v7_val_proba.npy')

print('Grid search pesos ensemble (D34)...')
print('  Espacio: w_RF ∈ [0.2..0.8 paso 0.1], w_XGB+w_LGB reparten el resto equitativamente')

best_qwk   = -1.0
best_rec   = -1.0
best_w     = None
best_cand  = False
results_gs = []

w_rf_values = np.arange(0.2, 0.91, 0.1)
w_xgb_frac  = [0.25, 0.33, 0.40, 0.50, 0.60, 0.67, 0.75]

for w_rf in w_rf_values:
    rem = 1.0 - w_rf
    for f_xgb in w_xgb_frac:
        w_xgb = round(rem * f_xgb, 4)
        w_lgb = round(rem * (1 - f_xgb), 4)
        w_lgb = 1.0 - w_rf - w_xgb

        proba = w_rf * p_rf + w_xgb * p_xgb + w_lgb * p_lgb
        y_pred = np.argmax(proba, axis=1)

        qwk = cohen_kappa_score(y_val, y_pred, weights='quadratic')
        n_alta = (y_val == 3).sum()
        tp_alta = ((y_pred == 3) & (y_val == 3)).sum()
        rec_alta = tp_alta / n_alta if n_alta > 0 else 0.0

        cand = qwk >= 0.65 and rec_alta >= 0.55
        results_gs.append(dict(w_rf=w_rf, w_xgb=w_xgb, w_lgb=w_lgb,
                               qwk=qwk, rec_alta=rec_alta, cand=cand))
        if qwk > best_qwk:
            best_qwk, best_rec, best_w, best_cand = qwk, rec_alta, (w_rf, w_xgb, w_lgb), cand

print(f'\nMejor combinación encontrada:')
print(f'  w_RF={best_w[0]:.2f}  w_XGB={best_w[1]:.2f}  w_LGB={best_w[2]:.2f}')
print(f'  QWK={best_qwk:.4f}  RecAlta={best_rec:.4f}  Candidato={best_cand}')

import pandas as pd
df_gs = pd.DataFrame(results_gs).sort_values('qwk', ascending=False)
print('\nTop 10 por QWK:')
print(df_gs.head(10).to_string(index=False, float_format='{:.4f}'.format))

candidatos_gs = df_gs[df_gs['cand']].head(5)
if len(candidatos_gs):
    print(f'\n✅ {len(df_gs[df_gs["cand"]])} combinaciones candidatas H2:')
    print(candidatos_gs.to_string(index=False, float_format='{:.4f}'.format))
else:
    print('\n❌ Ninguna combinación de pesos alcanza H2')

w_rf, w_xgb, w_lgb = best_w
proba_best = w_rf * p_rf + w_xgb * p_xgb + w_lgb * p_lgb
np.save(f'{OUT_DIR}/ens_best_v7_val_proba.npy', proba_best)

res_ens_best = evaluar(
    f'Ensemble óptimo (w_RF={w_rf:.2f} w_XGB={w_xgb:.2f} w_LGB={w_lgb:.2f})',
    y_val, proba_best)

if MLFLOW_OK:
    with mlflow.start_run(run_name='ens_grid_v7'):
        mlflow.log_params(dict(w_rf=w_rf, w_xgb=w_xgb, w_lgb=w_lgb,
                               method='grid_search_val', decision='D34'))
        mlflow.log_metrics(dict(qwk=res_ens_best['qwk'],
                                recall_alta=res_ens_best['recall_alta'],
                                macro_f1=res_ens_best['macro_f1'],
                                bal_acc=res_ens_best['bal_acc']))


Grid search pesos ensemble (D34)...
  Espacio: w_RF ∈ [0.2..0.8 paso 0.1], w_XGB+w_LGB reparten el resto equitativamente

Mejor combinación encontrada:
  w_RF=0.80  w_XGB=0.05  w_LGB=0.15
  QWK=0.6453  RecAlta=0.6685  Candidato=False

Top 10 por QWK:
  w_rf  w_xgb  w_lgb    qwk  rec_alta  cand
0.8000 0.0500 0.1500 0.6453    0.6685 False
0.8000 0.1000 0.1000 0.6453    0.6700 False
0.7000 0.0990 0.2010 0.6451    0.6644 False
0.8000 0.1200 0.0800 0.6451    0.6700 False
0.8000 0.0660 0.1340 0.6451    0.6689 False
0.7000 0.0750 0.2250 0.6451    0.6637 False
0.8000 0.0800 0.1200 0.6450    0.6690 False
0.7000 0.1200 0.1800 0.6450    0.6655 False
0.8000 0.1340 0.0660 0.6449    0.6700 False
0.8000 0.1500 0.0500 0.6449    0.6701 False

❌ Ninguna combinación de pesos alcanza H2

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  Ensemble óptimo (w_RF=0.80 w_XGB=0.05 w_LGB=0.15) — Validacion
  QWK=0.6453  MacroF1=0.6605  BalAcc=0.6779  

In [8]:
modelos_v7 = {
    'xgb_v7 (D32-sample_weight)': res_xgb,
    'lgb_v7 (D33-sample_weight)': res_lgb,
    'ens_grid_v7 (D34)':          res_ens_best,
}

sep = '=' * 118
print(f'\n{sep}')
print(f'  TABLA COMPARATIVA VALIDACION — v5 / v6 / v7')
print(f'{sep}')
hdr = f'  {"Modelo":<45} {"QWK":>8} {"RecAlta":>9} {"MacroF1":>9} {"BalAcc":>8}  H2?'
print(hdr); print(f'  {"-"*116}')
for nombre, r in refs.items():
    cand = r['qwk'] >= 0.65 and r['recall_alta'] >= 0.55
    print(f'  {nombre:<45} {r["qwk"]:>8.4f} {r["recall_alta"]:>9.4f}'
          f' {r["macro_f1"]:>9.4f} {r["bal_acc"]:>8.4f}  {"✅" if cand else "❌"}')
print(f'  {"─"*116}')
for nombre, r in modelos_v7.items():
    cand = r['qwk'] >= 0.65 and r['recall_alta'] >= 0.55
    print(f'  {nombre:<45} {r["qwk"]:>8.4f} {r["recall_alta"]:>9.4f}'
          f' {r["macro_f1"]:>9.4f} {r["bal_acc"]:>8.4f}  {"✅ CANDIDATO" if cand else "❌"}')
print(f'{sep}')
print(f'  Floor H2: QWK >= 0.65 AND RecAlta >= 0.55 (congelado — Kerr 1998)')
print(f'{sep}')

all_r = {**{k: dict(qwk=v['qwk'], recall_alta=v['recall_alta']) for k,v in refs.items()},
         **{k: dict(qwk=v['qwk'], recall_alta=v['recall_alta']) for k,v in modelos_v7.items()}}
candidatos = [(k,v) for k,v in all_r.items() if v['qwk']>=0.65 and v['recall_alta']>=0.55]
mejor = max(modelos_v7.items(), key=lambda x: x[1]['qwk'])

print()
if candidatos:
    print('✅ H2 VERIFICADA EN VAL v7')
    print(f'   Candidatos: {[c[0] for c in candidatos]}')
    print()
    print('   PROTOCOLO ONE-SHOT (Kaufman et al. 2012):')
    print('   → Crear 06 evaluacion test set v7.ipynb')
    print('   → Evaluar test set UNA SOLA VEZ con el mejor candidato en val')
else:
    print(f'❌ H2 NO VERIFICADA EN VAL v7 (0 candidatos)')
    print(f'   Mejor QWK v7 val: {mejor[1]["qwk"]:.4f} (modelo: {mejor[0]})')
    print(f'   Gap al floor: +{0.65 - mejor[1]["qwk"]:.4f} QWK')
    print()
    print('   Acciones siguientes (LOG-17 §17.5):')
    print('   1. Ordinal loss / QWK loss para XGB/LGB (de la Torre et al. 2018 CVPR)')
    print('   2. RF con 300-400 estimadores — único modelo que llega a QWK=0.6420')
    print('   3. Threshold tuning: bajar umbral de clase Alta en RF (sacrifica precisión)')
    print('   4. Revisar HORIZON=1 (30 min) — reduce ventaja del baseline de persistencia')

with open(f'{OUT_DIR}/resultados_v7_val.pkl', 'wb') as f:
    pickle.dump({k: {m: v for m,v in r.items() if m != 'confusion_matrix'}
                 for k,r in modelos_v7.items()}, f)
print(f'\nResultados v7 guardados: {OUT_DIR}/resultados_v7_val.pkl')



  TABLA COMPARATIVA VALIDACION — v5 / v6 / v7
  Modelo                                             QWK   RecAlta   MacroF1   BalAcc  H2?
  --------------------------------------------------------------------------------------------------------------------
  rf_v5                                           0.6420    0.6726    0.6560   0.6720  ❌
  xgb_v5                                          0.6232    0.6648    0.6511   0.6985  ❌
  lgb_v5                                          0.6300    0.6824    0.6533   0.7025  ❌
  ens_rfxgb_v6                                    0.6413    0.5605    0.6563   0.6164  ❌
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  xgb_v7 (D32-sample_weight)                      0.6227    0.6445    0.6503   0.6908  ❌
  lgb_v7 (D33-sample_weight)                      0.6299    0.6300    0.6543   0.6832  ❌
  ens_grid_v7 (D34)                               0.6453    0.6685    0.6605   0.6779  ❌
 